# Build an AI agent for an ordinary student task

The presentation uses Maya, Noah and Priya to explain prompts, context, tools and agent systems. This notebook begins with a completely new build:

- **Aisha** is revising a fictional commerce elective called **Leadership in Organisations**.
- Her agent reads simulated mastery data, searches supplied notes and chooses suitable study support.
- You write prompts; the Python loop and tools are already provided and remain inspectable.

The runtime records visible messages, tool requests, tool results, final answers and a quality check—not hidden reasoning.

> All course material, mastery data and student details in this exercise are fictional workshop simulations. The coach supports formative revision and must not write assessed responses.

## Setup

Run these cells once. Your API key is requested securely and is not written into the notebook.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from getpass import getpass
from dotenv import load_dotenv
import os

load_dotenv('.env', override=True)
if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Workshop Anthropic API key: ")

from workshopkit import *
print("Ready. Model:", DEFAULT_MODEL)

---
## Mission 1 — Build an adaptive revision coach

Aisha has ten days before her Leadership in Organisations exam. Her coach should inspect her mastery record, retrieve only the relevant supplied notes, and then decide whether she needs an explanation, flashcards, application questions or a combination.

You will edit exactly three prompt fields. The tools and agent loop remain fixed.

In [ ]:
# Inspect the prebuilt tool contracts
for tool in STUDY_COACH_TOOLS:
    print(f"\n{tool.name}: {tool.description}")
    print(tool.api_definition()["input_schema"])

In [ ]:
# YOUR TURN — edit these three prompts only
AGENT_INSTRUCTIONS = """
You are Aisha's adaptive revision coach for the fictional Leadership in Organisations unit.
Inspect the requested mastery profile before choosing study support. Retrieve the supplied
notes before making factual course claims. Use study-generation tools only for topics that
need practice, name the source section, and explain why you chose each activity. Support
formative revision; never write an assessed response or invent course material.
"""

STUDENT_REQUEST = """
Use Aisha's mixed mastery profile. Prepare a focused 25-minute study pack for the weakest
topic. Include the kind of practice the mastery result justifies, not every available tool.
"""

QUALITY_PROMPT = """
Check that the pack targets the weakest topic, is grounded in leadership_revision_notes.txt,
keeps one concept per flashcard, uses application rather than recall-only practice where
appropriate, labels simulation data, and does not create assessed work.
"""

study_result = run_study_coach(AGENT_INSTRUCTIONS, STUDENT_REQUEST, QUALITY_PROMPT)
show_trace(study_result, "AISHA'S ADAPTIVE STUDY COACH")

### Read the trace

- Did the agent inspect mastery before selecting an activity?
- Did it retrieve the relevant section before drafting course-specific material?
- Did it avoid tools that the result did not justify?
- Did the final quality check pass, or identify a concrete revision?

Compare your trace with someone nearby. Change one prompt field and run it again.

In [ ]:
# The simulator includes weak, mixed and strong profiles
for profile in ("weak", "mixed", "strong"):
    record = get_mastery_record.execute({"profile": profile})
    print(profile, record["scores"])

---
## Mission 2 — Decide whether Priya needs an orchestrator

Priya owns the argument and submitted writing. The specialists may identify relevant evidence and ask rubric-based revision questions. Each specialist is a separate model call, so delegation should earn its cost.

In [ ]:
for specialist in SPECIALISTS:
    print(f"{specialist.name}: {specialist.description}")

In [ ]:
# YOUR TURN
MANAGER_PROMPT = """
You coordinate revision support for a university student.
Delegate only when a specialist adds distinct value.
Do not write assessed prose. Preserve Priya's claims and ask her to decide
how to respond to evidence or rubric feedback.
Explain which specialists were useful before giving the final revision plan.
"""

PRIYA_TASK = """
I am arguing that access to primary sources changed how historians studied
student movements. My outline has sections on archives, oral histories and
digital collections. Help me identify what evidence I still need and test
the outline against a rubric requiring a defensible argument, relevant
evidence and clear distinction between evidence and interpretation.
"""

priya_result = run_manager(MANAGER_PROMPT, PRIYA_TASK, max_steps=6)
show_trace(priya_result, "PRIYA'S REVISION WORKFLOW")

### Architecture checkpoint

Would one careful model call have produced the same learning outcome? Compare usefulness, latency, number of calls and places the system could fail.

---
## Mission 3 — Retrieve private workshop documents

The assistant must search the fictional assessment, unit, room and calendar documents rather than guessing. It should name the files it relied on and say when they do not answer.

In [ ]:
print(search_student_docs.execute({"query": "extension deadline assessed work", "top_k": 3}))

In [ ]:
# YOUR TURN
DOCUMENT_INSTRUCTIONS = """
Answer questions about the supplied fictional university documents.
Retrieve when the answer depends on private workshop material.
Name the source files used, distinguish policy from advice and say when
the documents do not answer. Never describe workshop data as official UWA policy.
"""

DOCUMENT_TASK = """
Maya may need an extension for the history essay. What does the supplied
policy say she should do, and what information must she not invent?
"""

document_result = run_agent(
    DOCUMENT_INSTRUCTIONS,
    DOCUMENT_TASK,
    tools=[search_student_docs],
    max_steps=5,
)
show_trace(document_result, "DOCUMENT-AWARE ASSISTANT")

---
## Adapt the pattern for your own agent

Start with the smallest architecture that can work:

1. What student outcome matters?
2. What context is needed now?
3. Which facts require retrieval or a tool?
4. Is the path fixed, or must a model choose the next step?
5. Which actions require a person to approve?
6. How will you check that the system supports learning rather than replacing it?

In [ ]:
# YOUR TURN
MY_INSTRUCTIONS = """You are ..."""
MY_TASK = """..."""
MY_TOOLS = []  # Add only tools the task genuinely needs

my_result = run_agent(MY_INSTRUCTIONS, MY_TASK, tools=MY_TOOLS, max_steps=6)
show_trace(my_result, "MY STUDENT ASSISTANT")